In [1]:
import os
import pandas as pd
import numpy as np
import dblp
from crossref.restful import Works
import requests
from serpapi import GoogleSearch

In [143]:
crossref_work = Works()

In [17]:
raw_dfs = [
    "data/cleaned/social/search-results-master-v3.csv",
]

In [141]:
# Prepare DataFrame to store the results
columns = [
    "PaperTitle",
    "DOI",
    "Authors",
    "Abstract",
    "Publisher",
    "SemanticScholarUrl",
    "DoiUrl",
    "PublicationDate",
    "FieldOfStudy",
    "Conference-Journal",
    "PublicationTypes",
    "SearchString",
    "CitationCount",
    "SearchedFrom",
]

In [142]:
sch_fields = [
    "title",
    "externalIds",
    "authors",
    "abstract",
    "url",
    "publicationDate",
    "fieldsOfStudy",
    "venue",
    "publicationTypes",
    "citationCount",
    "externalIds",
]

In [144]:
# Helpers
def extract_sch(api_url, sch_fields):
    headers = {"Content-Type": "application/json"}
    params = {
        "fields": ",".join(sch_fields),
    }
    try:
        response = requests.get(api_url, headers=headers, params=params)
        result = response.json()
        return result
    except Exception as e:
        print(e)
        return {"error": e}


def get_affiliations_google_scholar(author_name) -> list[str]:
    params = {
        "engine": "google_scholar_profiles",
        "mauthors": author_name.strip(),
        "api_key": os.environ.get("GOOGLE_SCHOLAR_API_KEY"),
    }

    try:
        search = GoogleSearch(params)
        results = search.get_dict()
        if (
            results.get("search_metadata", {}).get("status") == "Error"
            or len(results.get("profiles", [])) == 0
        ):
            return ["No Affiliation"]
        else:
            return [results["profiles"][0]["affiliations"]]
    except Exception as e:
        print(e)
        return ["No Affiliation"]


def extract_authors(sch_paper, crossref_paper):
    authors = None
    if crossref_paper is not None:
        authors = crossref_paper.get("author")
    if authors is not None:
        for i in range(len(authors)):
            author = authors[i]
            author_name = author.get("given", "") + " " + author.get("family", "")
            affiliations = author.get("affiliation", [])
            school_names = (
                [affil.get("name") for affil in affiliations]
                if affiliations != []
                else ["No Affiliation"]
            )
            # Create a new dictionary with only 'name' and 'affiliation'
            authors[i] = {
                "name": author_name.strip(),
                "affiliation": school_names,
            }
    else:
        authors = sch_paper.get("authors")
        for i in range(len(authors)):
            author = authors[i]
            affiliations = author.get("affiliations")
            if affiliations is None:
                author_name = author["name"]
                affiliations = ["No Affiliation"]

            authors[i] = {
                "name": author_name.strip(),
                "affiliation": affiliations,
            }

    return authors

In [197]:
def extract_data(row):
    pid = row["ID"]
    if "URL" in pid and "abs-" in pid:
        pid = "arXiv:" + pid.split("abs-")[1].replace("-", ".")

    api_url = f"https://api.semanticscholar.org/graph/v1/paper/{pid}"

    sch_paper = extract_sch(api_url, sch_fields)

    if sch_paper.get("error") is not None:
        # if can't search with ID, search by paper title

        api_url = f"https://api.semanticscholar.org/graph/v1/paper/search?query={row['PaperTitle']}"

        sch_paper = extract_sch(api_url, sch_fields)

        if len(sch_paper.get("data", [])) == 0:
            return {
                "PaperTitle": row["PaperTitle"],
                "DOI": pid,
                "Authors": None,
                "Abstract": None,
                "Publisher": None,
                "SemanticScholarUrl": None,
                "DoiUrl": None,
                "PublicationDate": None,
                "FieldOfStudy": None,
                "Conference-Journal": None,
                "PublicationTypes": None,
                "SearchString": row["SearchString"],
                "CitationCount": None,
                "SearchedFrom": row["SearchedFrom"],
            }
        else:
            sch_paper = sch_paper["data"][0]

    doi = sch_paper.get("externalIds", {}).get("DOI", None)

    if doi is None and str(row["ID"]).startswith("DOI"):
        doi = row["ID"].split(":")[1]

    try:
        crossref_paper = crossref_work.doi(doi)

    except Exception as e:
        crossref_paper = None

    title = row["PaperTitle"]

    authors = extract_authors(sch_paper, crossref_paper)

    abstract = sch_paper.get("abstract", None)

    sch_url = sch_paper.get("url", None)

    doi_url = f"https://doi.org/{doi}" if doi is not None else None

    publication_date = sch_paper.get("publicationDate", None)

    fields_of_study = sch_paper.get("fieldsOfStudy", [])

    venue = sch_paper.get("venue", None)

    # publisher

    if crossref_paper is not None:
        publisher = crossref_paper.get("publisher")

    elif doi and "arxiv" in doi.lower():
        publisher = "arXiv"
    else:
        publisher = None

    # paper type

    if crossref_paper is not None and crossref_paper.get("type") is not None:
        paper_type = [crossref_paper.get("type")]
    else:
        paper_type = sch_paper.get("publicationTypes", [])
        if paper_type is not None:
            paper_type = ["".join(t.split("-")).lower() for t in paper_type]

    citation_count = sch_paper.get("citationCount", None)

    # TODO: paper keywords missing

    # TODO: paper type is conference/journal for arxiv papers

    # TODO: conference-journal name mismatch with publisher, i.e., for paper with name"ChatGPT in education: A discourse analysis of worries and concerns on social media", the conference name is "International Conference on Artificial Intelligence in Education", but the publisher is "Arxiv" (becauseit queryed from arxiv), need "Springer" instead.

    new_paper = {
        "PaperTitle": title,
        "DOI": doi,
        "Authors": authors,
        "Abstract": abstract,
        "Publisher": publisher,
        "SemanticScholarUrl": sch_url,
        "DoiUrl": doi_url,
        "PublicationDate": publication_date,
        "FieldOfStudy": fields_of_study,
        "Conference-Journal": venue,
        "PublicationTypes": paper_type,
        "SearchString": row["SearchString"],
        "CitationCount": citation_count,
        "SearchedFrom": row["SearchedFrom"],
    }

    return new_paper

In [18]:
for raw_df_path in raw_dfs:
    raw_df = pd.read_csv(raw_df_path)
    total_rows = len(raw_df)
    results = []
    
    # raw_df = raw_df.iloc[2571:]

    for index, row in raw_df.iterrows():
        # if index >= 5:
        #     break
        print(f"Processing {raw_df_path}: row {index + 1}/{total_rows}...")
        paper = extract_data(row)
        results.append(paper)

    # Creating a DataFrame from the results
    results_df = pd.DataFrame(results, columns=columns)

    # Generating a new file name based on the raw data file name
    new_file_name = raw_df_path.replace(".csv", "-full.csv")
    # new_file_name = 'test.csv'

    # Saving to a CSV file
    results_df.to_csv(new_file_name, index=False)

Processing data/cleaned/social/search-results-master-v3.csv: row 1/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 2/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 3/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 4/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 5/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 6/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 7/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 8/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 9/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 10/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 11/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 12/2066...
Processing data/cleaned/social/search-results-master-v3.csv: row 13/2066...
Processing data/clean

# Post-processing stuff

In [204]:
filename = "./data/cleaned/edu/search-results-master-v3-full.csv"

In [205]:
df_clean = pd.read_csv(filename)
df_clean

,PaperTitle,DOI,Authors,Abstract,Publisher,SemanticScholarUrl,DoiUrl,PublicationDate,FieldOfStudy,Conference-Journal,PublicationTypes,SearchString,CitationCount,SearchedFrom,ID
0,Can we use ChatGPT for Mental Health and Subst...,10.2196/51243,"[{'name': 'Sophia Spallek', 'affiliation': ['N...",Background The use of generative artificial in...,NaN,https://www.semanticscholar.org/paper/083509bb...,https://doi.org/10.2196/51243,2023-07-26,['Medicine'],JMIR Medical Education,['journalarticle'],"""Generative Artificial Intelligence"" + ""misinf...",0.0,Semantic Scholar,NaN
1,Ethical implications of ChatGPT in higher educ...,10.48550/arXiv.2311.14378,"[{'name': 'Ming Li', 'affiliation': ['Northwes...",This scoping review explores the ethical chall...,arXiv,https://www.semanticscholar.org/paper/7611de3f...,https://doi.org/10.48550/arXiv.2311.14378,2023-11-24,['Computer Science'],arXiv.org,"['journalarticle', 'review']","""Generative Artificial Intelligence"" + ""misinf...",1.0,Semantic Scholar,NaN
2,Engineering Education in the Era of ChatGPT: P...,10.1109/EDUCON54358.2023.10125121,"[{'name': 'Junaid Qadir', 'affiliation': ['Pro...",Engineering education is constantly evolving t...,NaN,https://www.semanticscholar.org/paper/d553d008...,https://doi.org/10.1109/EDUCON54358.2023.10125121,2023-05-01,['Computer Science'],IEEE Global Engineering Education Conference,"['journalarticle', 'conference', 'review']","""Generative Artificial Intelligence"" + ""misinf...",154.0,Semantic Scholar,NaN
3,Developing authentic assessment through open e...,10.14742/apubs.2023.657,"[{'name': 'Mais Fatayer', 'affiliation': ['Uni...",This Pecha Kucha presentation will showcase Th...,NaN,https://www.semanticscholar.org/paper/1e0f6f05...,https://doi.org/10.14742/apubs.2023.657,2023-11-28,NaN,ASCILITE Publications,['journalarticle'],"""Generative Artificial Intelligence"" + ""access...",0.0,Semantic Scholar,NaN
4,Artificial intelligence in developing countrie...,10.1177/02666669231200628,"[{'name': 'Nishith Reddy Mannuru', 'affiliatio...",This paper explores the potential impact of Ge...,NaN,https://www.semanticscholar.org/paper/5a5e03c3...,https://doi.org/10.1177/02666669231200628,2023-09-14,NaN,Information Development,['journalarticle'],"""Generative Artificial Intelligence"" + ""access...",3.0,Semantic Scholar,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,The Covid-19 pandemic: An opportunity for refl...,10.17159/2224-7912/2020/V60N4-2A1,"[{'name': 'B. Olivier', 'affiliation': ['Penn ...","The present article poses the question, whethe...",NaN,https://www.semanticscholar.org/paper/f7362485...,https://doi.org/10.17159/2224-7912/2020/V60N4-2A1,NaN,['Political Science'],NaN,NaN,"""PaLM"" ""responsible"" ""education""",1.0,Web of Science,NaN
571,Studying the effect of AI Code Generators on S...,10.1145/3544548.3580919,"[{'name': 'Majeed Kazemitabaar', 'affiliation'...",AI code generators like OpenAI Codex have the ...,NaN,https://www.semanticscholar.org/paper/f03f39f7...,https://doi.org/10.1145/3544548.3580919,2023-02-15,['Computer Science'],International Conference on Human Factors in C...,"['journalarticle', 'book', 'conference']","""GPT-3"" ""over-reliance"" ""education""",53.0,Web of Science,NaN
572,Artificial Hallucinations by Google Bard: Thin...,10.7759/cureus.43313,"[{'name': 'Mukesh Kumar', 'affiliation': ['API...",One of the critical challenges posed by artifi...,NaN,https://www.semanticscholar.org/paper/0049f577...,https://doi.org/10.7759/cureus.43313,2023-08-01,['Medicine'],Cureus,"['journalarticle', 'editorial']","""GPT"" ""misinformation"" ""education""",4.0,Web of Science,NaN
573,Exploring the role of ChatGPT in patient care ...,10.1101/2023.06.13.23291311,"[{'name': 'R. Garg', 'affiliation': ['PostDoct...",Background ChatGPT(Chat Generative Pre-trained...,NaN,https://www.semanticscholar.org/paper/1c2a3841...,https://doi.org/10.1101/2023.06.13.23291311,2023-06-14,['Medicine'],medRxiv,"['journalarticle', 'review']","""GPT"

In [206]:
# drop column 'ID'
df_clean = df_clean.drop(columns=["ID"])

In [154]:
def cast_affliation(authors):
  if authors == np.nan or type(authors) != str:
    return authors
  print(authors)
  authors = eval(authors)
  for i in range(len(authors)):
    author = dict(authors[i])
    if type(author["affiliation"]) is str:
      print(author["affiliation"])
      author["affiliation"] = [author["affiliation"]]
    authors[i] = author
  return authors

In [155]:
df_clean["Authors"] = df_clean["Authors"].apply(lambda x: cast_affliation(x))

[{'name': 'Sophia Spallek', 'affiliation': ['No Affiliation']}, {'name': 'Louise Birrell', 'affiliation': 'The Matilda Centre, The University of Sydney'}, {'name': 'Stephanie Kershaw', 'affiliation': ['No Affiliation']}, {'name': 'Emma Krogh Devine', 'affiliation': ['No Affiliation']}, {'name': 'Louise Thornton', 'affiliation': 'University of Sydney'}]
The Matilda Centre, The University of Sydney
University of Sydney
[{'name': 'Ming Li', 'affiliation': 'Northwest A & F University, China'}, {'name': 'Ariunaa Enkhtur', 'affiliation': 'Osaka University'}, {'name': 'Fei Cheng', 'affiliation': 'The Hong Kong University of Science and Technology'}, {'name': 'B. Yamamoto', 'affiliation': ''}]
Northwest A & F University, China
Osaka University
The Hong Kong University of Science and Technology

[{'name': 'Junaid Qadir', 'affiliation': ['College of Engineering, Qatar University,Department of Computer Science and Engineering,Doha,Qatar']}]
[{'name': 'Mais Fatayer', 'affiliation': 'University of 

In [156]:
df_clean["Authors"]

0       [{'name': 'Sophia Spallek', 'affiliation': ['N...
1       [{'name': 'Ming Li', 'affiliation': ['Northwes...
2       [{'name': 'Junaid Qadir', 'affiliation': ['Col...
3       [{'name': 'Mais Fatayer', 'affiliation': ['Uni...
4       [{'name': 'Nishith Reddy Mannuru', 'affiliatio...
                              ...                        
2934                                                  NaN
2935                                                  NaN
2936                                                  NaN
2937                                                  NaN
2938                                                  NaN
Name: Authors, Length: 2939, dtype: object

In [159]:
df_clean["DoiUrl"] = df_clean["DoiUrl"].apply(
    lambda x: x if x != np.nan and type(x) == str and "None" not in x else None
)

In [201]:
df_clean["PublicationTypes"] = df_clean["PublicationTypes"].apply(
    lambda x: ["".join(t.split("-")).lower() for t in eval(x)]
    if x != np.nan and type(x) == str
    else x
)

In [207]:
df_clean

,PaperTitle,DOI,Authors,Abstract,Publisher,SemanticScholarUrl,DoiUrl,PublicationDate,FieldOfStudy,Conference-Journal,PublicationTypes,SearchString,CitationCount,SearchedFrom
0,Can we use ChatGPT for Mental Health and Subst...,10.2196/51243,"[{'name': 'Sophia Spallek', 'affiliation': ['N...",Background The use of generative artificial in...,NaN,https://www.semanticscholar.org/paper/083509bb...,https://doi.org/10.2196/51243,2023-07-26,['Medicine'],JMIR Medical Education,['journalarticle'],"""Generative Artificial Intelligence"" + ""misinf...",0.0,Semantic Scholar
1,Ethical implications of ChatGPT in higher educ...,10.48550/arXiv.2311.14378,"[{'name': 'Ming Li', 'affiliation': ['Northwes...",This scoping review explores the ethical chall...,arXiv,https://www.semanticscholar.org/paper/7611de3f...,https://doi.org/10.48550/arXiv.2311.14378,2023-11-24,['Computer Science'],arXiv.org,"['journalarticle', 'review']","""Generative Artificial Intelligence"" + ""misinf...",1.0,Semantic Scholar
2,Engineering Education in the Era of ChatGPT: P...,10.1109/EDUCON54358.2023.10125121,"[{'name': 'Junaid Qadir', 'affiliation': ['Pro...",Engineering education is constantly evolving t...,NaN,https://www.semanticscholar.org/paper/d553d008...,https://doi.org/10.1109/EDUCON54358.2023.10125121,2023-05-01,['Computer Science'],IEEE Global Engineering Education Conference,"['journalarticle', 'conference', 'review']","""Generative Artificial Intelligence"" + ""misinf...",154.0,Semantic Scholar
3,Developing authentic assessment through open e...,10.14742/apubs.2023.657,"[{'name': 'Mais Fatayer', 'affiliation': ['Uni...",This Pecha Kucha presentation will showcase Th...,NaN,https://www.semanticscholar.org/paper/1e0f6f05...,https://doi.org/10.14742/apubs.2023.657,2023-11-28,NaN,ASCILITE Publications,['journalarticle'],"""Generative Artificial Intelligence"" + ""access...",0.0,Semantic Scholar
4,Artificial intelligence in developing countrie...,10.1177/02666669231200628,"[{'name': 'Nishith Reddy Mannuru', 'affiliatio...",This paper explores the potential impact of Ge...,NaN,https://www.semanticscholar.org/paper/5a5e03c3...,https://doi.org/10.1177/02666669231200628,2023-09-14,NaN,Information Development,['journalarticle'],"""Generative Artificial Intelligence"" + ""access...",3.0,Semantic Scholar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,The Covid-19 pandemic: An opportunity for refl...,10.17159/2224-7912/2020/V60N4-2A1,"[{'name': 'B. Olivier', 'affiliation': ['Penn ...","The present article poses the question, whethe...",NaN,https://www.semanticscholar.org/paper/f7362485...,https://doi.org/10.17159/2224-7912/2020/V60N4-2A1,NaN,['Political Science'],NaN,NaN,"""PaLM"" ""responsible"" ""education""",1.0,Web of Science
571,Studying the effect of AI Code Generators on S...,10.1145/3544548.3580919,"[{'name': 'Majeed Kazemitabaar', 'affiliation'...",AI code generators like OpenAI Codex have the ...,NaN,https://www.semanticscholar.org/paper/f03f39f7...,https://doi.org/10.1145/3544548.3580919,2023-02-15,['Computer Science'],International Conference on Human Factors in C...,"['journalarticle', 'book', 'conference']","""GPT-3"" ""over-reliance"" ""education""",53.0,Web of Science
572,Artificial Hallucinations by Google Bard: Thin...,10.7759/cureus.43313,"[{'name': 'Mukesh Kumar', 'affiliation': ['API...",One of the critical challenges posed by artifi...,NaN,https://www.semanticscholar.org/paper/0049f577...,https://doi.org/10.7759/cureus.43313,2023-08-01,['Medicine'],Cureus,"['journalarticle', 'editorial']","""GPT"" ""misinformation"" ""education""",4.0,Web of Science
573,Exploring the role of ChatGPT in patient care ...,10.1101/2023.06.13.23291311,"[{'name': 'R. Garg', 'affiliation': ['PostDoct...",Background ChatGPT(Chat Generative Pre-trained...,NaN,https://www.semanticscholar.org/paper/1c2a3841...,https://doi.org/10.1101/2023.06.13.23291311,2023-06-14,['Medicine'],medRxiv,"['journalarticle', 'review']","""GPT"" ""privacy"" ""education""",7.0,Web of

In [209]:
df_raw = pd.read_csv("./data/cleaned/social/search-results-master-v2.csv")
df_clean = pd.read_csv("./data/cleaned/social/search-results-master-v2-full.csv")

In [208]:
df_clean.to_csv(filename, index=False) 

In [214]:
pd.merge(df_raw, df_clean, on=["PaperTitle", "SearchString", "SearchedFrom"], how="left").drop_duplicates(subset=["PaperTitle"], keep="last").to_csv("./data/cleaned/social/search-results-master-v2-full.csv", index=False)

# Testing stuff

In [37]:
len(results)

179

In [14]:
api_url = "https://api.semanticscholar.org/graph/v1/paper/search/"
headers = {
    "Content-Type": "application/json",
    "x-api-key": "X48LIBLqr86ouHlnMYd3z052sgEm3Nd2wMORPzu5",
}
fields = ["title", "externalIds", "paperId", "url"]
sch_search_string = "Saving the Vicuna: The Political, Biophysical, and Cultural History of Wild Animal Conservation in Peru, 1964-2000"
params = {"query": sch_search_string, "year": "2019-", "fields": ",".join(fields)}
response = requests.get(api_url, headers=headers, params=params)
result = response.json()
print(result)

{'total': 1, 'offset': 0, 'data': [{'paperId': '64fa0be92833e344e79f205976fb77a489461a58', 'externalIds': {'MAG': '3005763097', 'DOI': '10.1093/ahr/rhz939', 'CorpusId': 213377959}, 'url': 'https://www.semanticscholar.org/paper/64fa0be92833e344e79f205976fb77a489461a58', 'title': 'Saving the Vicuña: The Political, Biophysical, and Cultural History of Wild Animal Conservation in Peru, 1964–2000'}]}


In [12]:
pid = "ArXiv:2205.07144"
api_url = f"https://api.semanticscholar.org/graph/v1/paper/{pid}"
sch_paper = extract_sch(api_url, sch_fields)
sch_paper

{'paperId': 'cb179c916e73c7a6b0edf973b4ed087fe6224722',
 'externalIds': {'DBLP': 'conf/nips/LiBY22',
  'ArXiv': '2205.07144',
  'CorpusId': 248811155},
 'url': 'https://www.semanticscholar.org/paper/cb179c916e73c7a6b0edf973b4ed087fe6224722',
 'title': 'Network change point localisation under local differential privacy',
 'abstract': 'Network data are ubiquitous in our daily life, containing rich but often sensitive information. In this paper, we expand the current static analysis of privatised networks to a dynamic framework by considering a sequence of networks with potential change points. We investigate the fundamental limits in consistently localising change points under both node and edge privacy constraints, demonstrating interesting phase transition in terms of the signal-to-noise ratio condition, accompanied by polynomial-time algorithms. The private signal-to-noise ratio conditions quantify the costs of the privacy for change point localisation problems and exhibit a different

In [ ]:
# merge the three new csv, on paper title and doi